In [ ]:
# explode() Flattening Array Columns

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col

spark = SparkSession.builder.appName("ExplodeExample").getOrCreate()

data = [
    (1, "Alice", ["Laptop", "Mouse", "Keyboard"]),
    (2, "Bob", ["Phone", "Charger"]),
    (3, "Carlos", ["Tablet"]),
    (4, "David", [])
]

df = spark.createDataFrame(data, ["cust_id", "name", "items"])
df.show(truncate=False)


+-------+------+-------------------------+
|cust_id|name  |items                    |
+-------+------+-------------------------+
|1      |Alice |[Laptop, Mouse, Keyboard]|
|2      |Bob   |[Phone, Charger]         |
|3      |Carlos|[Tablet]                 |
|4      |David |[]                       |
+-------+------+-------------------------+



In [ ]:
df_exploded = df.select("cust_id", "name", explode("items").alias("item"))
df_exploded.show()


+-------+------+--------+
|cust_id|  name|    item|
+-------+------+--------+
|      1| Alice|  Laptop|
|      1| Alice|   Mouse|
|      1| Alice|Keyboard|
|      2|   Bob|   Phone|
|      2|   Bob| Charger|
|      3|Carlos|  Tablet|
+-------+------+--------+



In [ ]:
# posexplode() — Flatten with Position Index posexplode() works like explode() but also returns the position index of each element in the array.

from pyspark.sql.functions import posexplode

df_pos = df.select("cust_id", "name", posexplode("items").alias("pos", "item")) # The pos column represents array element index (starting at 0).
df_pos.show()


+-------+------+---+--------+
|cust_id|  name|pos|    item|
+-------+------+---+--------+
|      1| Alice|  0|  Laptop|
|      1| Alice|  1|   Mouse|
|      1| Alice|  2|Keyboard|
|      2|   Bob|  0|   Phone|
|      2|   Bob|  1| Charger|
|      3|Carlos|  0|  Tablet|
+-------+------+---+--------+



# repartition() vs coalesce()

# repartition(n) → increases/decreases partitions via shuffle

# coalesce(n) → reduces partitions without shuffle (efficient for output)

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RepartitionVsCoalesce").getOrCreate()

# Sample dataset with 100 numbers
data = [(i,) for i in range(1, 101)]
df = spark.createDataFrame(data, ["num"])


In [2]:
# Check Initial Partition Count

print("Initial partitions:", df.rdd.getNumPartitions())


Initial partitions: 2


In [3]:
# Using repartition() — Causes Shuffle

df_repart = df.repartition(15)  # data is shuffled across 5 partitions for better parallelism.
print("After repartition:", df_repart.rdd.getNumPartitions())


After repartition: 15


In [ ]:
df_repart.explain(True)

== Parsed Logical Plan ==
Repartition 5, true
+- LogicalRDD [num#234L], false

== Analyzed Logical Plan ==
num: bigint
Repartition 5, true
+- LogicalRDD [num#234L], false

== Optimized Logical Plan ==
Repartition 5, true
+- LogicalRDD [num#234L], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ShuffleQueryStage 0
   +- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [plan_id=295]
      +- *(1) Scan ExistingRDD[num#234L]
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [plan_id=290]
   +- Scan ExistingRDD[num#234L]



In [4]:
# Using coalesce() — No Shuffle

df_coal = df_repart.coalesce(2)
print("After coalesce:", df_coal.rdd.getNumPartitions()) # park merged existing partitions without shuffling data around — this is efficient when reducing partitions.


After coalesce: 2


In [ ]:
# Partitioning by Column

from pyspark.sql.functions import col

df_country = spark.createDataFrame([
    ("Alice", "US"), ("Bob", "US"), ("Carlos", "BR"),
    ("David", "IN"), ("Eva", "IN"), ("Frank", "BR")
], ["name", "country"])

df_part = df_country.repartition(3, col("country"))
print("Repartitioned by country:", df_part.rdd.getNumPartitions())



Repartitioned by country: 3


# A User Defined Function (UDF) lets you write custom Python logic that Spark can apply to columns in a DataFrame.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("UDF Example").getOrCreate()

data = [
    (1, "Alice", 5000),
    (2, "Bob", 4000),
    (3, "Charlie", 10000),
    (4, "David", 3000)
]
df = spark.createDataFrame(data, ["id", "name", "salary"])

df.show()


+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  1|  Alice|  5000|
|  2|    Bob|  4000|
|  3|Charlie| 10000|
|  4|  David|  3000|
+---+-------+------+



# Define a Simple Python Function

In [ ]:
def bonus_calculator(salary):
    return salary * 0.10  # 10% bonus

# Register UDF with Return Type

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

bonus_udf = udf(bonus_calculator, DoubleType())

# Apply to DataFrame:

In [ ]:
df_bonus = df.withColumn("bonus", bonus_udf(df["salary"]))
df_bonus.show()


+---+-------+------+------+
| id|   name|salary| bonus|
+---+-------+------+------+
|  1|  Alice|  5000| 500.0|
|  2|    Bob|  4000| 400.0|
|  3|Charlie| 10000|1000.0|
|  4|  David|  3000| 300.0|
+---+-------+------+------+



# UDF with multiple columns

In [ ]:
def remark(name, salary):
    if salary >= 8000:
        return f"{name} is a high earner"
    else:
        return f"{name} is a regular employee"

In [ ]:
from pyspark.sql.types import StringType

remark_udf = udf(remark, StringType())

df_remark = df.withColumn("remark", remark_udf(df["name"], df["salary"]))
df_remark.show(truncate=False)

+---+-------+------+---------------------------+
|id |name   |salary|remark                     |
+---+-------+------+---------------------------+
|1  |Alice  |5000  |Alice is a regular employee|
|2  |Bob    |4000  |Bob is a regular employee  |
|3  |Charlie|10000 |Charlie is a high earner   |
|4  |David  |3000  |David is a regular employee|
+---+-------+------+---------------------------+



# Registering as SQL UDF   use UDFs inside Spark SQL queries.


In [ ]:
spark.udf.register("bonus_sql", bonus_calculator, DoubleType())

df.createOrReplaceTempView("employees")

spark.sql("""
    SELECT id, name, salary, bonus_sql(salary) AS bonus
    FROM employees
""").show()


+---+-------+------+------+
| id|   name|salary| bonus|
+---+-------+------+------+
|  1|  Alice|  5000| 500.0|
|  2|    Bob|  4000| 400.0|
|  3|Charlie| 10000|1000.0|
|  4|  David|  3000| 300.0|
+---+-------+------+------+

